In [0]:
from pyspark.sql.functions import col, sum, avg, count, round, current_timestamp

print("🏆 Initializing Gold Layer processing for data analytics aggregates...")

# 1. Read your polished customer/claim logs from the Silver layer
silver_data = spark.read.table("claim_investigation_analysis.02_silver.claim_notes_cleaned")

# =========================================================================
# GOLD METRIC AGGREGATION: Risk Segment Business Intelligence
# =========================================================================
# Calculating volumetric sizes, total financial values, and averages per segment
gold_risk_metrics = (silver_data
    .groupBy("risk_classification")
    .agg(
        count("claim_id").alias("total_claims_filed"),
        # We cast the claim string tokens to numeric types to mock monetary assets
        round(sum(col("claim_id").cast("double")), 2).alias("total_financial_exposure"),
        round(avg(col("claim_id").cast("double")), 2).alias("average_claim_cost")
    )
    .withColumn("metrics_updated_at", current_timestamp())
)

# 2. Write out the final business metrics summary into your 03_gold database schema
(gold_risk_metrics.write
    .format("delta")
    .mode("overwrite") # Keeps our batch runs perfectly clean and updated
    .saveAsTable("claim_investigation_analysis.03_gold.summary_risk_metrics"))

print("📊 Gold metrics successfully aggregated and published to 03_gold.summary_risk_metrics!")

# =========================================================================
# 3. DISPLAY THE OUTPUT METRICS
# =========================================================================
print("\n👀 Previewing the completed Gold Layer analytics table:")
display(gold_risk_metrics)
